# TRIBE v2 — Cognitive Load Demo on Merge Conflicts

Demonstração mínima de uso do TRIBE v2 (Meta, 2026) para estimar carga cognitiva induzida por blocos de conflito de merge.

**Como rodar no Colab Free:**
1. Upload da pasta `tribe/` inteira (via painel lateral → Files).
2. Runtime → Change runtime type → GPU (T4).
3. Executar células **em ordem**.
4. **OBRIGATÓRIO:** após a célula de install, reiniciar o runtime (Runtime → Restart session) antes de seguir.

## 1. Install — clonar repo + alinhar torch/torchaudio

O pyproject do TRIBE pina `torch<2.7`, o que rebaixa o torch padrão do Colab mas não rebaixa o `torchaudio`. Sem alinhar manualmente, o import falha silenciosamente.

In [ ]:
# Clone com nome diferente para evitar shadow do pacote
!rm -rf /content/tribev2_repo
!git clone -q https://github.com/facebookresearch/tribev2.git /content/tribev2_repo

In [ ]:
%cd /content/tribev2_repo
!pip install -q -e ".[plotting]"
%cd /content

In [ ]:
# Alinha torchaudio com o torch que o tribev2 fixou (>=2.5.1,<2.7).
# --no-deps impede pip de voltar o torch pra 2.10.
!pip install -q "torchaudio>=2.5.1,<2.7" --no-deps --force-reinstall

## 2. REINICIAR O RUNTIME AGORA

**Runtime → Restart session** (ou Ctrl+M .). Depois pular a célula de install e continuar daqui.

Sem reiniciar, o torch antigo continua carregado em memória e os imports vão falhar.

## 3. Verificar GPU

In [ ]:
import torch
print("torch:", torch.__version__)
import torchaudio
print("torchaudio:", torchaudio.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 4. Import diagnóstico

Se algo quebrar no import, esta célula mostra a stack trace real (em vez da mensagem enganosa `unknown location`).

In [ ]:
import sys, traceback

# Garante que /content (que contém a pasta tribev2_repo) não sombreie o pacote instalado
sys.path = [p for p in sys.path if p not in ("", "/content")]
for mod in list(sys.modules):
    if mod.startswith("tribev2"):
        del sys.modules[mod]

try:
    import tribev2
    print("tribev2.__file__:", tribev2.__file__)
    from tribev2 import TribeModel
    print("TribeModel imported OK")
except Exception:
    print("=== REAL ERROR BELOW ===")
    traceback.print_exc()

## 5. Carregar o modelo

Primeira execução baixa os pesos (alguns GB). Subsequentes usam cache local da sessão.

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from tribev2 import TribeModel

# Autenticação no Hugging Face.
# TRIBE puxa internamente meta-llama/Llama-3.2-3B (repo gated do Meta).
# Pré-requisitos OBRIGATÓRIOS na sua conta HF (não dá pra resolver no código):
#   1) Aceitar os termos em https://huggingface.co/meta-llama/Llama-3.2-3B
#      (formulário com nome/afiliação). Confirme que aparece como "Accepted"
#      em https://huggingface.co/settings/gated-repos.
#   2) Gerar um token com permissão Read em
#      https://huggingface.co/settings/tokens e salvar como secret HF_TOKEN
#      no Colab (ícone de chave no painel lateral, com "Notebook access" ON).
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

assert hf_token, "HF_TOKEN não encontrado. Defina o secret HF_TOKEN no Colab (ícone de chave)."

from huggingface_hub import login, whoami, hf_hub_download
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

login(token=hf_token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

# Sanity check 1: o token é válido?
me = whoami(token=hf_token)
print(f"HF user: {me.get('name')} ({me.get('type')})")

# Sanity check 2: a conta foi aprovada para o Llama 3.2?
# Falha cedo aqui em vez de gastar 4min em TTS antes do 403.
try:
    hf_hub_download(
        repo_id="meta-llama/Llama-3.2-3B",
        filename="config.json",
        token=hf_token,
    )
    print("Acesso a meta-llama/Llama-3.2-3B: OK")
except GatedRepoError as e:
    raise RuntimeError(
        "Sua conta HF ainda NÃO foi aprovada para meta-llama/Llama-3.2-3B.\n"
        "Vá em https://huggingface.co/meta-llama/Llama-3.2-3B, preencha o\n"
        "formulário de acesso, e confirme em\n"
        "https://huggingface.co/settings/gated-repos que o status é 'Accepted'.\n"
        "Confirme também que o HF_TOKEN é desta mesma conta."
    ) from e

model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")
print("Model loaded.")

## 6. Listar conflitos de teste

In [ ]:
# Ajusta o caminho se a pasta conflicts/ estiver em outro lugar no Colab.
conflict_dir = Path("/content/conflicts")
if not conflict_dir.exists():
    # fallback: pasta no diretório atual
    conflict_dir = Path("conflicts")

conflicts = sorted(conflict_dir.glob("*.txt"))
print(f"Found {len(conflicts)} conflict file(s) in {conflict_dir}:")
for c in conflicts:
    print(f"  - {c.name} ({c.stat().st_size} bytes)")

## 7. Predição de carga cognitiva por conflito

Para cada conflito, o TRIBE converte texto em fala internamente, processa pelos encoders (LLaMA 3.2, Wav2Vec-BERT) e prediz a resposta fMRI na malha cortical fsaverage5 (~20k vértices).

**Proxy de carga (versão crua):**
- `mean_load` = magnitude média da ativação predita.
- `peak_load` = maior média de ativação em um único timestep.

In [ ]:
results = []
predictions = {}  # guarda preds/segments por conflito para a análise da seção 9
for path in conflicts:
    print(f"\nProcessing {path.name}...")
    df_events = model.get_events_dataframe(text_path=str(path))
    preds, segments = model.predict(events=df_events)
    predictions[path.stem] = (preds, segments)
    mean_load = float(np.mean(np.abs(preds)))
    peak_load = float(np.max(np.mean(np.abs(preds), axis=1)))
    results.append({
        "conflict": path.stem,
        "timesteps": int(preds.shape[0]),
        "vertices": int(preds.shape[1]),
        "mean_load": mean_load,
        "peak_load": peak_load,
    })
    print(f"  shape: {preds.shape}")
    print(f"  mean_load: {mean_load:.4f}")
    print(f"  peak_load: {peak_load:.4f}")

## 8. Comparar resultados

In [ ]:
df_results = pd.DataFrame(results).sort_values("mean_load", ascending=False).reset_index(drop=True)
df_results

In [ ]:
df_results.to_csv("tribe_results.csv", index=False)
print("Saved to tribe_results.csv — faça download via painel lateral.")

## 9. Análise detalhada de uma predição

A saída do TRIBE é uma matriz `preds` de shape `(timesteps, vertices)` — uma série temporal de respostas BOLD preditas em ~20k vértices da malha cortical fsaverage5 (LH + RH concatenados). Cada timestep corresponde a um TR (típico fMRI ≈ 1.49s).

Aqui descompomos a predição de **um conflito** em três dimensões:
- **Espacial**: quais vértices acumularam mais ativação ao longo do tempo (`vertex_mean`).
- **Temporal**: em que momento da leitura o cérebro foi mais ativado (`time_mean`).
- **Eventos**: quais palavras/tokens estavam sendo processados nos picos temporais.

In [ ]:
# Escolha qual conflito inspecionar. Default: o de maior mean_load.
target = df_results.iloc[0]["conflict"]
# target = "scenario_38"  # descomente para fixar

preds, segments = predictions[target]
print(f"Analisando conflito: {target}")
print(f"  preds.shape       : {preds.shape}  (timesteps, vertices)")
print(f"  segments (palavras): {len(segments)}")

# Reduções
vertex_mean = np.mean(np.abs(preds), axis=0)   # (n_vertices,)  — carga total por vértice
time_mean   = np.mean(np.abs(preds), axis=1)   # (n_timesteps,) — carga global por timestep

top_v_idx = np.argsort(vertex_mean)[-10:][::-1]
print("\nTop-10 vértices mais ativados:")
for i, v in enumerate(top_v_idx, 1):
    hemi = "LH" if v < preds.shape[1] // 2 else "RH"
    print(f"  {i:2d}. vertex={v:5d}  hemi={hemi}  load={vertex_mean[v]:.4f}")

peak_t = int(np.argmax(time_mean))
print(f"\nPico temporal: t={peak_t} (carga={time_mean[peak_t]:.4f})")
if hasattr(segments, "__len__") and len(segments) > peak_t:
    print(f"  segmento no pico: {segments[peak_t]!r}")

In [ ]:
# Curva temporal: carga média por timestep ao longo da leitura do conflito.
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(time_mean, lw=1.2)
ax.axvline(peak_t, color="red", ls="--", lw=0.8, label=f"pico t={peak_t}")
ax.set_xlabel("timestep (TR ~ 1.49s)")
ax.set_ylabel("carga média |BOLD| (todos vértices)")
ax.set_title(f"Carga cognitiva ao longo da leitura — {target}")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Mapa cortical — onde a carga incide

Renderizamos `vertex_mean` (carga acumulada por vértice ao longo do tempo) sobre a malha **fsaverage5** usando `nilearn`. TRIBE concatena hemisférios na ordem `[LH | RH]`, então cortamos pela metade.

- **Visão estática (`plot_surf_stat_map`)**: 4 vistas (lateral/medial × LH/RH).
- **Visão interativa (`view_surf`)**: cérebro 3D girável dentro do notebook.

Regiões esperadas para leitura de código (literatura de neurociência de programação, Siegmund 2014 / Floyd 2017): giro frontal inferior esquerdo (Broca), giro temporal médio, sulco intraparietal — todos relacionados a linguagem + working memory.

In [ ]:
from nilearn import datasets, plotting

fsaverage = datasets.fetch_surf_fsaverage("fsaverage5")  # 10242 vértices por hemisfério

n_vert = preds.shape[1]
half = n_vert // 2
lh_load = vertex_mean[:half]
rh_load = vertex_mean[half:]
print(f"LH: {lh_load.shape}  RH: {rh_load.shape}")

# Threshold ~percentil 75 destaca só vértices acima da carga típica.
thr = float(np.percentile(vertex_mean, 75))
print(f"Threshold (p75): {thr:.4f}")

fig, axes = plt.subplots(2, 2, figsize=(12, 8), subplot_kw={"projection": "3d"})
for ax, (hemi, mesh, bg, data, view) in zip(
    axes.flat,
    [
        ("left",  fsaverage.infl_left,  fsaverage.sulc_left,  lh_load, "lateral"),
        ("left",  fsaverage.infl_left,  fsaverage.sulc_left,  lh_load, "medial"),
        ("right", fsaverage.infl_right, fsaverage.sulc_right, rh_load, "lateral"),
        ("right", fsaverage.infl_right, fsaverage.sulc_right, rh_load, "medial"),
    ],
):
    plotting.plot_surf_stat_map(
        mesh, data, hemi=hemi, view=view,
        bg_map=bg, threshold=thr, colorbar=True,
        cmap="hot", axes=ax, title=f"{hemi.upper()} {view}",
    )
fig.suptitle(f"Carga cognitiva por vértice — {target}", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Versão interativa (3D girável). Renderiza inline no notebook;
# .save_as_html(...) gera arquivo standalone para anexar no TCC.
view_lh = plotting.view_surf(
    surf_mesh=fsaverage.infl_left,
    surf_map=lh_load,
    bg_map=fsaverage.sulc_left,
    threshold=thr,
    cmap="hot",
    title=f"LH — {target}",
)
view_lh.save_as_html(f"brain_lh_{target}.html")
view_lh

In [ ]:
view_rh = plotting.view_surf(
    surf_mesh=fsaverage.infl_right,
    surf_map=rh_load,
    bg_map=fsaverage.sulc_right,
    threshold=thr,
    cmap="hot",
    title=f"RH — {target}",
)
view_rh.save_as_html(f"brain_rh_{target}.html")
view_rh

## 11. Interpretação

**Se a hipótese for confirmada** (`scenario_38` > `01-method-conflict`): TRIBE diferencia dificuldade neurocognitiva entre tipos de conflito.

**Se inconclusivo ou invertido**: investigar:
- Normalizar carga por número de timesteps (scenario_38 é maior).
- Agregação global mascara sinais específicos — testar ROIs frontoparietais (Yeo-7, ver `nilearn.datasets.fetch_atlas_yeo_2011`).
- Conversão texto→fala pode descaracterizar código — considerar input visual.

**Localização anatômica esperada** (validação qualitativa do mapa cortical):
- *Hit* na literatura: giro frontal inferior esquerdo, giro temporal médio, sulco intraparietal.
- *Miss*: ativação difusa ou predominante em córtex visual primário sugere que o modelo está respondendo à fala TTS, não ao conteúdo semântico do conflito.